# 10. External testing integrations

Advanced maintainer notebook for Rhesis and Semantica evaluation. This is optional and not part of the required live 3-hour participant flow.

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiment' / 'run.py').exists():
    candidates = [Path('/content/kdd'), Path('/content/drive/MyDrive/kdd'), Path('/Users/syaikhipin/Documents/Phd/Publish Paper/kdd')]
    PROJECT_ROOT = next((p for p in candidates if (p / 'experiment' / 'run.py').exists()), PROJECT_ROOT)

EXPERIMENT_DIR = PROJECT_ROOT / 'experiment'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
sys.path.insert(0, str(EXPERIMENT_DIR))
print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /Users/syaikhipin/Documents/Phd/Publish Paper/kdd


## Integration availability

Rhesis requires `rhesis-sdk` and `RHESIS_API_KEY` in the environment. Semantica requires the `semantica` package. The notebook only checks presence by default.

In [2]:
availability = {
    'rhesis_sdk_importable': importlib.util.find_spec('rhesis') is not None,
    'rhesis_api_key_present': bool(os.environ.get('RHESIS_API_KEY')),
    'semantica_importable': importlib.util.find_spec('semantica') is not None,
}
print(json.dumps(availability, indent=2))

{
  "rhesis_sdk_importable": true,
  "rhesis_api_key_present": false,
  "semantica_importable": false
}


## Offline semantic evaluation

This deterministic path is safe for CI, Colab, and tutorial participants.

In [3]:
cmd = [
    sys.executable, str(EXPERIMENT_DIR / 'run.py'),
    '--mode', 'real',
    '--backend', 'offline',
    '--datasets', 'locomo', 'longmemeval', 'memoryarena',
    '--longmemeval-files', 'longmemeval_oracle.json',
    '--max-conversations', '2',
    '--max-questions', '20',
    '--max-items', '100',
    '--top-k', '5',
    '--eval-backend', 'offline',
    '--eval-limit', '50',
]
print(' '.join(map(str, cmd)))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=240)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0

/Users/syaikhipin/anaconda3/bin/python /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/run.py --mode real --backend offline --datasets locomo longmemeval memoryarena --longmemeval-files longmemeval_oracle.json --max-conversations 2 --max-questions 20 --max-items 100 --top-k 5 --eval-backend offline --eval-limit 50


dataset	strategy	questions	retr_p	retr_r	hit	util	lat_ms	cost	mem	sem_cov	sem	sem_pass	faith	ctx	ans
LoCoMo	no_memory	40	0.000	0.000	0.000	0.000	0.00	4.0	0	0.100	0.000	0.000	0.000	0.000	0.000
LoCoMo	verbatim	40	0.070	0.321	0.350	0.350	0.86	23.8	369	0.100	0.400	0.250	0.250	0.699	0.250
LoCoMo	extracted_facts	40	0.075	0.346	0.375	0.375	0.86	23.8	369	0.100	0.385	0.250	0.250	0.653	0.250
LoCoMo	episodic	40	0.080	0.371	0.400	0.400	0.92	23.8	369	0.100	0.400	0.250	0.250	0.699	0.250
LoCoMo	hybrid	40	0.080	0.371	0.400	0.400	0.87	23.8	369	0.100	0.413	0.250	0.250	0.741	0.250
LongMemEval	no_memory	100	0.000	0.000	0.000	0.000	0.00	10.0	0	0.030	0.000	0.000	0.000	0.000	0.000
LongMemEval	verbatim	100	1.000	0.998	1.000	1.000	0.05	15.5	5	0.030	1.000	1.000	1.000	1.000	1.000
LongMemEval	extracted_facts	100	1.000	0.998	1.000	1.000	0.04	15.5	5	0.030	1.000	1.000	1.000	1.000	1.000
LongMemEval	episodic	100	1.000	0.998	1.000	1.000	0.05	15.5	5	0.030	1.000	1.000	1.000	1.000	1.000
LongMemEval	hybrid	100	1.000	0.998	

## Optional Rhesis smoke test

Set `RUN_RHESIS_SMOKE=1` and provide `RHESIS_API_KEY` outside the notebook to run this.

In [4]:
if os.environ.get('RUN_RHESIS_SMOKE') == '1':
    cmd = [sys.executable, str(EXPERIMENT_DIR / 'run.py'), '--mode', 'real', '--eval-backend', 'rhesis', '--eval-smoke-test', '--eval-limit', '1']
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=240)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print('Skipped Rhesis smoke test.')

Skipped Rhesis smoke test.


## Optional Semantica smoke test

In [5]:
if availability['semantica_importable']:
    cmd = [sys.executable, str(EXPERIMENT_DIR / 'run.py'), '--mode', 'real', '--eval-backend', 'semantica', '--eval-smoke-test', '--eval-limit', '1']
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=240)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print('Skipped Semantica smoke test because semantica is not importable.')

Skipped Semantica smoke test because semantica is not importable.


## Optional Modal GPU runner

Use this only for maintainer or paper-quality runs. Keep Modal credentials in the environment and set `RUN_MODAL_SMOKE=1` to execute a tiny remote GPU smoke test.

In [6]:
if os.environ.get('RUN_MODAL_SMOKE') == '1':
    cmd = [
        sys.executable, str(EXPERIMENT_DIR / 'run.py'),
        '--runner', 'modal',
        '--modal-gpu', os.environ.get('MODAL_GPU', 'T4'),
        '--modal-detach',
        '--mode', 'real',
        '--datasets', 'locomo',
        '--max-conversations', '1',
        '--max-questions', '1',
        '--eval-backend', 'offline',
        '--eval-smoke-test',
        '--visualize',
    ]
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=3600)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print('Skipped Modal GPU smoke test.')
    print('Detached Modal runs print a call id; fetch with --modal-call-id <call-id>.')

Skipped Modal GPU smoke test.
Detached Modal runs print a call id; fetch with --modal-call-id <call-id>.


In [7]:
latest = sorted(RESULTS_DIR.glob('run_*_real_metrics.json'))[-1]
metrics = json.loads(latest.read_text())['real']
for dataset, summary in metrics['datasets'].items():
    for strategy, row in summary['by_strategy'].items():
        if 'mean_semantic_score' in row:
            print(dataset, strategy, row['semantic_coverage_rate'], row['mean_semantic_score'], row['semantic_pass_rate'])

LoCoMo no_memory 0.1 0.0 0.0
LoCoMo verbatim 0.1 0.3996 0.25
LoCoMo extracted_facts 0.1 0.3845 0.25
LoCoMo episodic 0.1 0.3996 0.25
LoCoMo hybrid 0.1 0.4135 0.25
LongMemEval no_memory 0.03 0.0 0.0
LongMemEval verbatim 0.03 1.0 1.0
LongMemEval extracted_facts 0.03 1.0 1.0
LongMemEval episodic 0.03 1.0 1.0
LongMemEval hybrid 0.03 1.0 1.0
MemoryArena no_memory 0.0062 0.0 0.0
MemoryArena verbatim 0.0062 0.0 0.0
MemoryArena extracted_facts 0.0062 1.0 1.0
MemoryArena episodic 0.0062 0.0 0.0
MemoryArena hybrid 0.0062 0.0 0.0
